# Shifu OCR — Colab Training v2
Uses the ACTUAL engine code (177 unified features: static + perturbation + shape).

**Upload two files:**
1. `shifu_engine.zip` — the engine code (21 KB)
2. `training_data.zip` — the training images (257 MB)

In [ ]:
!pip install numpy Pillow scipy scikit-image -q

In [ ]:
# Upload engine code + training data
from google.colab import files
import os, zipfile

print('Upload shifu_engine.zip AND training_data.zip')
uploaded = files.upload()

for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('.')
        print(f'Extracted {name}')

# Find training dir
TRAINING_DIR = 'training_data'
if os.path.exists('training_data/training_data'):
    TRAINING_DIR = 'training_data/training_data'
img_dir = os.path.join(TRAINING_DIR, 'images')
n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
print(f'Found {n_images} training images')

# Verify engine
from shifu_ocr.engine import ShifuOCR, extract_features
import numpy as np
ocr = ShifuOCR()
test_img = np.full((100,100), 255, dtype=np.uint8)
test_img[30:70, 40:60] = 0
fv = ocr._extract_unified_features(test_img)
print(f'Feature dimensions: {len(fv)} (should be 177)')

In [ ]:
# Train with multiprocessing
import time
import multiprocessing as mp
from PIL import Image
from collections import defaultdict

ALLOWED_CHARS = set('abcdefghijklmnopqrstuvwxyz0123456789')

def process_image(args):
    img_path, label = args
    try:
        from shifu_ocr.engine import ShifuOCR
        _ocr = ShifuOCR()  # lightweight — just for feature extraction
        img = Image.open(img_path)
        if img.mode != 'L': img = img.convert('L')
        arr = np.array(img)
        img.close()
        segments = _ocr.segment_characters(arr, min_char_width=2)
        if not segments: return []
        clean_chars = [c.lower() for c in label if c.lower() in ALLOWED_CHARS]
        if not clean_chars: return []
        ratio = len(segments) / max(len(clean_chars), 1)
        if ratio < 0.8 or ratio > 1.2: return []
        n = min(len(segments), len(clean_chars))
        results = []
        for i in range(n):
            fv = _ocr._extract_unified_features(segments[i]['image'])
            results.append((clean_chars[i], fv.tolist()))
        return results
    except:
        return []

# Load entries
entries = []
with open(os.path.join(TRAINING_DIR, 'train_list.txt'), 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or '\t' not in line: continue
        img_path, label = line.split('\t', 1)
        full_path = os.path.join(TRAINING_DIR, img_path)
        if os.path.exists(full_path):
            entries.append((full_path, label))
print(f'Loaded {len(entries)} images')

n_cores = mp.cpu_count()
print(f'Using {n_cores} cores')

from shifu_ocr.engine import Landscape
landscapes = {}
total_chars = 0
start = time.time()
batch_size = 2000

for batch_start in range(0, len(entries), batch_size):
    batch = entries[batch_start:batch_start + batch_size]
    with mp.Pool(n_cores) as pool:
        results = pool.map(process_image, batch)
    for char_list in results:
        for label, fv_list in char_list:
            fv = np.array(fv_list)
            if label not in landscapes:
                landscapes[label] = Landscape(label)
            landscapes[label].absorb(fv)
            total_chars += 1
    elapsed = time.time() - start
    bn = batch_start // batch_size + 1
    tb = (len(entries) + batch_size - 1) // batch_size
    print(f'  Batch {bn}/{tb}: {total_chars} chars, {total_chars/max(elapsed,1):.0f}/sec, {elapsed:.0f}s')

print(f'\nDone: {total_chars} chars in {(time.time()-start)/60:.1f} min')
print(f'Landscapes: {len(landscapes)}')

In [ ]:
# Save model
import json
model = {
    'version': '2.0.0',
    'total_predictions': 0,
    'total_correct': 0,
    'landscapes': {k: v.to_dict() for k, v in landscapes.items()},
}
with open('trained_model.json', 'w') as f:
    json.dump(model, f)
size_mb = os.path.getsize('trained_model.json') / 1024 / 1024
print(f'Saved: {size_mb:.1f} MB')

# Verify dimensions
first = list(landscapes.values())[0]
print(f'Feature dim: {len(first.mean)} (must be 177)')

In [ ]:
# Validation
val_entries = []
vp = os.path.join(TRAINING_DIR, 'val_list.txt')
if os.path.exists(vp):
    with open(vp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line: continue
            img_path, label = line.split('\t', 1)
            full_path = os.path.join(TRAINING_DIR, img_path)
            if os.path.exists(full_path):
                val_entries.append((full_path, label))

print(f'Validating on {min(len(val_entries), 300)} images...')
ocr_test = ShifuOCR()
ocr_test.landscapes = landscapes
correct = 0
tested = 0
for img_path, label in val_entries[:300]:
    try:
        img = Image.open(img_path).convert('L')
        result = ocr_test.read_line(np.array(img))
        img.close()
        pred = result['text'].lower().replace(' ','')
        truth = ''.join(c.lower() for c in label if c.lower() in ALLOWED_CHARS)
        for a, b in zip(pred, truth):
            tested += 1
            if a == b: correct += 1
    except: pass
print(f'Accuracy: {correct}/{tested} ({100*correct/max(tested,1):.1f}%)')

In [ ]:
# Download
from google.colab import files
files.download('trained_model.json')